In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 220)

DB_DIR = Path(os.environ.get('SUJET2_DB_DIR', 'db'))
OUT_DIR = Path(os.environ.get('SUJET2_OUT_DIR', 'output'))
OUT_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    'sitehist': DB_DIR / 'sitehist.csv',
    'sitepred': DB_DIR / 'sitepred.csv',
    'siteweath': DB_DIR / 'siteweath.csv',
    'zonehist': DB_DIR / 'zonehist.csv',
    'zonepred': DB_DIR / 'zonepred.csv',
}

def read_semicolon_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=';', quotechar='"', encoding='utf-8', engine='python')

raw = {}
for k, p in paths.items():
    if p.exists():
        raw[k] = read_semicolon_csv(p)
        print(k, 'loaded', raw[k].shape, 'from', p)
    else:
        print(k, 'missing', p)

sitehist loaded (3492, 15) from db\sitehist.csv
sitepred loaded (3573, 9) from db\sitepred.csv
siteweath loaded (4184, 15) from db\siteweath.csv
zonehist loaded (21048, 16) from db\zonehist.csv
zonepred loaded (20237, 10) from db\zonepred.csv


In [2]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().strip('"') for c in df.columns]
    lower_map = {c.lower(): c for c in df.columns}

    def rename_if_present(target: str, *candidates: str):
        for cand in candidates:
            if cand in lower_map:
                df.rename(columns={lower_map[cand]: target}, inplace=True)
                break

    rename_if_present('siteId', 'siteid', 'site_id', 'site id')
    rename_if_present('zoneId', 'zoneid', 'zone_id', 'zone id')
    rename_if_present('dtUpdate', 'dtupdate', 'dt_update', 'dt update')
    return df


def to_day(s):
    return pd.to_datetime(s, errors='coerce').dt.floor('D')

sitehist = normalize_columns(raw.get('sitehist', pd.DataFrame()))
sitepred = normalize_columns(raw.get('sitepred', pd.DataFrame()))
siteweath = normalize_columns(raw.get('siteweath', pd.DataFrame()))
zonehist = normalize_columns(raw.get('zonehist', pd.DataFrame()))
zonepred = normalize_columns(raw.get('zonepred', pd.DataFrame()))

if 'dtUpdate' in sitehist.columns and 'date' not in sitehist.columns:
    sitehist['date'] = to_day(sitehist['dtUpdate'])
if 'date' in sitepred.columns:
    sitepred['date'] = to_day(sitepred['date'])
if 'dtUpdate' in siteweath.columns and 'date' not in siteweath.columns:
    siteweath['date'] = to_day(siteweath['dtUpdate'])
if 'dtUpdate' in zonehist.columns and 'date' not in zonehist.columns:
    zonehist['date'] = to_day(zonehist['dtUpdate'])
if 'date' in zonepred.columns:
    zonepred['date'] = to_day(zonepred['date'])

for name, df in [('sitehist', sitehist), ('sitepred', sitepred), ('siteweath', siteweath), ('zonehist', zonehist), ('zonepred', zonepred)]:
    cols = list(df.columns)
    print(name, 'cols', cols[:12], '... total', len(cols))
    if 'date' in df.columns:
        print(name, 'date dtype', df['date'].dtype)

sitehist cols ['id', 'siteId', 'indoorTempDegC', 'elecBveKwh', 'elecCvcKwh', 'elecForceKwh', 'elecLightingKwh', 'elecAggregatedKwh', 'elecTotalKwh', 'waterM3', 'maxUsers', 'totalOptimValue'] ... total 16
sitehist date dtype datetime64[ns]
sitepred cols ['id', 'siteId', 'value', 'totalKwh', 'tempAmb', 'totalWater', 'date', 'usrUpdate', 'dtUpdate'] ... total 9
sitepred date dtype datetime64[ns]
siteweath cols ['id', 'siteId', 'tempMin', 'tempMax', 'chancePluie', 'summary', 'djC', 'djF', 'indexApprox', 'weekend', 'airId', 'tempAmb'] ... total 16
siteweath date dtype datetime64[ns]
zonehist cols ['id', 'siteId', 'zoneId', 'indoorTempDegC', 'elecBveKwh', 'elecCvcKwh', 'elecForceKwh', 'elecLightingKwh', 'elecAggregatedKwh', 'elecTotalKwh', 'waterM3', 'maxUsers'] ... total 17
zonehist date dtype datetime64[ns]
zonepred cols ['id', 'siteId', 'zoneId', 'value', 'totalKwh', 'tempAmb', 'totalWater', 'date', 'usrUpdate', 'dtUpdate'] ... total 10
zonepred date dtype datetime64[ns]


In [3]:
ZERO_OR_NEG_IS_MISSING = {
    'elecTotalKwh': True,
    'totalKwh': True,
    'waterM3': True,
    'totalWater': True,
    'totalOptimValue': True,
    'value': True,
}

UNIT_FIX_RULES = {
    'elecTotalKwh': [1000, 1_000_000],
    'totalKwh': [1000, 1_000_000],
    'waterM3': [1000],
    'totalWater': [1000],
}

CUMUL_SPIKE = {
    'lookback_days': 10,
    'min_missing_run': 3,
    'spike_factor': 20.0,
    'strategy': 'spread',
}

MEASURE_COLS_SITEHIST = [c for c in ['elecTotalKwh', 'waterM3', 'totalOptimValue', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh'] if c in sitehist.columns]
MEASURE_COLS_ZONEHIST = [c for c in ['elecTotalKwh', 'waterM3', 'totalOptimValue', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh', 'indoorTempDegC'] if c in zonehist.columns]
MEASURE_COLS_SITEPRED = [c for c in ['totalKwh', 'totalWater', 'value', 'tempAmb'] if c in sitepred.columns]
MEASURE_COLS_ZONEPRED = [c for c in ['totalKwh', 'totalWater', 'value', 'tempAmb'] if c in zonepred.columns]

print('measure cols sitehist', MEASURE_COLS_SITEHIST)
print('measure cols sitepred', MEASURE_COLS_SITEPRED)
print('measure cols zonehist', MEASURE_COLS_ZONEHIST)
print('measure cols zonepred', MEASURE_COLS_ZONEPRED)

measure cols sitehist ['elecTotalKwh', 'waterM3', 'totalOptimValue', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh']
measure cols sitepred ['totalKwh', 'totalWater', 'value', 'tempAmb']
measure cols zonehist ['elecTotalKwh', 'waterM3', 'totalOptimValue', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh', 'indoorTempDegC']
measure cols zonepred ['totalKwh', 'totalWater', 'value', 'tempAmb']


In [4]:
def _num(s):
    return pd.to_numeric(s, errors='coerce')


def apply_missing_sentinels(df: pd.DataFrame, cols, rule_map: dict) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        x = _num(df[c])
        if rule_map.get(c, False):
            x = x.mask(x <= 0, np.nan)
        else:
            x = x.mask(x < 0, np.nan)
        df[c] = x
    return df


def fix_unit_outliers(df: pd.DataFrame, group_cols, value_col: str, divisors, factor=200.0, min_med=1e-9):
    df = df.copy()
    log = []
    if value_col not in df.columns:
        return df, pd.DataFrame(log)

    df[value_col] = _num(df[value_col])
    med = (df.loc[df[value_col].notna() & (df[value_col] > 0)]
             .groupby(group_cols)[value_col].median())
    med_map = med.to_dict()

    vals = df[value_col].to_numpy()
    new_vals = vals.copy()

    for i in range(len(df)):
        v = vals[i]
        if not np.isfinite(v) or v <= 0:
            continue
        k = tuple(df.iloc[i][group_cols].values.tolist())
        m = med_map.get(k, np.nan)
        if not np.isfinite(m) or m < min_med:
            continue
        if v / m <= factor:
            continue

        best = None
        best_ratio = v / m
        for d in divisors:
            vd = v / d
            rd = vd / m
            if rd < best_ratio:
                best_ratio = rd
                best = (d, vd)
        if best is not None and best_ratio <= factor:
            d, vd = best
            new_vals[i] = vd
            log.append({'value_col': value_col, 'group': k, 'index': int(i), 'old': float(v), 'new': float(vd), 'divisor': int(d)})

    df[value_col] = new_vals
    return df, pd.DataFrame(log)


def spread_cumul_spikes(df: pd.DataFrame, group_cols, date_col: str, value_col: str, cfg: dict):
    df = df.copy()
    logs = []
    if value_col not in df.columns or date_col not in df.columns:
        return df, pd.DataFrame(logs)

    df[value_col] = _num(df[value_col])

    lookback = int(cfg.get('lookback_days', 10))
    min_run = int(cfg.get('min_missing_run', 3))
    spike_factor = float(cfg.get('spike_factor', 20.0))
    strategy = cfg.get('strategy', 'spread')

    out_parts = []
    for keys, g in df.groupby(group_cols, dropna=False):
        g = g.sort_values(date_col)
        if g[date_col].isna().all():
            out_parts.append(g)
            continue

        idx = pd.date_range(g[date_col].min(), g[date_col].max(), freq='D')
        g2 = g.set_index(date_col).reindex(idx)
        g2.index.name = date_col

        if not isinstance(keys, tuple):
            keys = (keys,)
        for c, v in zip(group_cols, keys):
            g2[c] = v

        x = g2[value_col].copy()
        roll_med = x.rolling(window=max(lookback, 3), min_periods=3).median()

        is_missing = x.isna() | (x <= 0)
        run = np.zeros(len(x), dtype=int)
        for i in range(1, len(x)):
            run[i] = run[i - 1] + 1 if is_missing.iloc[i - 1] else 0

        for i in range(len(x)):
            v = x.iloc[i]
            if not np.isfinite(v) or v <= 0:
                continue
            if run[i] < min_run:
                continue
            m = roll_med.iloc[i]
            if not np.isfinite(m) or m <= 0:
                continue
            if v <= spike_factor * m:
                continue

            start = i - run[i]
            end = i
            span = end - start + 1

            if strategy == 'drop':
                x.iloc[i] = np.nan
                logs.append({'group': keys, 'value_col': value_col, 'date': str(g2.index[i].date()), 'action': 'drop', 'old': float(v), 'span_days': int(span)})
            else:
                per_day = v / span
                for j in range(start, end + 1):
                    if j == end or is_missing.iloc[j]:
                        x.iloc[j] = per_day
                logs.append({'group': keys, 'value_col': value_col, 'date': str(g2.index[i].date()), 'action': 'spread', 'old': float(v), 'new_per_day': float(per_day), 'span_days': int(span)})

        g2[value_col] = x
        out_parts.append(g2.reset_index())

    out = pd.concat(out_parts, ignore_index=True)
    out = out[[c for c in df.columns if c in out.columns]]
    return out, pd.DataFrame(logs)

In [5]:
CLEAN_LOGS = {}

sitehist = apply_missing_sentinels(sitehist, MEASURE_COLS_SITEHIST, ZERO_OR_NEG_IS_MISSING)
sitepred = apply_missing_sentinels(sitepred, MEASURE_COLS_SITEPRED, ZERO_OR_NEG_IS_MISSING)
zonehist = apply_missing_sentinels(zonehist, MEASURE_COLS_ZONEHIST, ZERO_OR_NEG_IS_MISSING)
zonepred = apply_missing_sentinels(zonepred, MEASURE_COLS_ZONEPRED, ZERO_OR_NEG_IS_MISSING)

for df in (sitehist, sitepred, siteweath, zonehist, zonepred):
    if 'siteId' in df.columns:
        df['siteId'] = pd.to_numeric(df['siteId'], errors='coerce').astype('Int64')
    if 'zoneId' in df.columns:
        df['zoneId'] = pd.to_numeric(df['zoneId'], errors='coerce').astype('Int64')
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.floor('D')

if set(['siteId']).issubset(sitehist.columns):
    for col, divs in UNIT_FIX_RULES.items():
        if col in sitehist.columns:
            sitehist, log = fix_unit_outliers(sitehist, ['siteId'], col, divs)
            if len(log):
                CLEAN_LOGS[f'sitehist_unit_{col}'] = log
if set(['siteId']).issubset(sitepred.columns):
    for col, divs in UNIT_FIX_RULES.items():
        if col in sitepred.columns:
            sitepred, log = fix_unit_outliers(sitepred, ['siteId'], col, divs)
            if len(log):
                CLEAN_LOGS[f'sitepred_unit_{col}'] = log

if set(['siteId', 'zoneId']).issubset(zonehist.columns):
    for col, divs in UNIT_FIX_RULES.items():
        if col in zonehist.columns:
            zonehist, log = fix_unit_outliers(zonehist, ['siteId', 'zoneId'], col, divs)
            if len(log):
                CLEAN_LOGS[f'zonehist_unit_{col}'] = log
if set(['siteId', 'zoneId']).issubset(zonepred.columns):
    for col, divs in UNIT_FIX_RULES.items():
        if col in zonepred.columns:
            zonepred, log = fix_unit_outliers(zonepred, ['siteId', 'zoneId'], col, divs)
            if len(log):
                CLEAN_LOGS[f'zonepred_unit_{col}'] = log

if set(['siteId', 'date']).issubset(sitehist.columns):
    for col in [c for c in MEASURE_COLS_SITEHIST if c in ('elecTotalKwh', 'waterM3', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh')]:
        sitehist, log = spread_cumul_spikes(sitehist, ['siteId'], 'date', col, CUMUL_SPIKE)
        if len(log):
            CLEAN_LOGS[f'sitehist_cumul_{col}'] = log

if set(['siteId', 'zoneId', 'date']).issubset(zonehist.columns):
    for col in [c for c in MEASURE_COLS_ZONEHIST if c in ('elecTotalKwh', 'waterM3', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh')]:
        zonehist, log = spread_cumul_spikes(zonehist, ['siteId', 'zoneId'], 'date', col, CUMUL_SPIKE)
        if len(log):
            CLEAN_LOGS[f'zonehist_cumul_{col}'] = log

print('clean logs', list(CLEAN_LOGS.keys()))

clean logs ['zonehist_unit_elecTotalKwh', 'zonehist_unit_waterM3', 'zonepred_unit_totalKwh', 'zonepred_unit_totalWater', 'sitehist_cumul_elecLightingKwh', 'zonehist_cumul_elecCvcKwh', 'zonehist_cumul_elecLightingKwh', 'zonehist_cumul_elecAggregatedKwh']


In [6]:
site_join_cols = ['siteId', 'date']
zone_join_cols = ['siteId', 'zoneId', 'date']

for req, df, name in [(site_join_cols, sitehist, 'sitehist'), (site_join_cols, sitepred, 'sitepred')]:
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise KeyError(f'{name} missing {missing}. columns={list(df.columns)[:20]}')

sitehist_d = sitehist.dropna(subset=site_join_cols).drop_duplicates(subset=site_join_cols, keep='last')
sitepred_d = sitepred.dropna(subset=site_join_cols).drop_duplicates(subset=site_join_cols, keep='last')

site_base = sitepred_d.merge(sitehist_d, on=site_join_cols, how='left', suffixes=('_pred', '_hist'))

if len(siteweath) and set(site_join_cols).issubset(siteweath.columns):
    siteweath_d = siteweath.dropna(subset=site_join_cols).drop_duplicates(subset=site_join_cols, keep='last')
    w = siteweath_d.rename(columns={c: f'weath_{c}' for c in siteweath_d.columns if c not in site_join_cols})
    site_base = site_base.merge(w, on=site_join_cols, how='left')

for req, df, name in [(zone_join_cols, zonehist, 'zonehist'), (zone_join_cols, zonepred, 'zonepred')]:
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise KeyError(f'{name} missing {missing}. columns={list(df.columns)[:20]}')

zonehist_d = zonehist.dropna(subset=zone_join_cols).drop_duplicates(subset=zone_join_cols, keep='last')
zonepred_d = zonepred.dropna(subset=zone_join_cols).drop_duplicates(subset=zone_join_cols, keep='last')

zone_base = zonepred_d.merge(zonehist_d, on=zone_join_cols, how='left', suffixes=('_pred', '_hist'))

print('site_base', site_base.shape)
print('zone_base', zone_base.shape)

site_base (3572, 37)
zone_base (20237, 24)


In [7]:
def mae(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.mean(np.abs(y_true[m] - y_pred[m]))) if m.any() else np.nan


def rmse(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(np.mean((y_true[m] - y_pred[m]) ** 2))) if m.any() else np.nan


def mape(y_true, y_pred, eps=1e-9):
    m = np.isfinite(y_true) & np.isfinite(y_pred) & (np.abs(y_true) > eps)
    return float(100.0 * np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m]))) if m.any() else np.nan


def bias(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.mean(y_pred[m] - y_true[m])) if m.any() else np.nan


def metrics_row(df, y_true_col, y_pred_col):
    y_true_obj = df[y_true_col]
    y_pred_obj = df[y_pred_col]
    if isinstance(y_true_obj, pd.DataFrame):
        y_true_obj = y_true_obj.iloc[:, 0]
    if isinstance(y_pred_obj, pd.DataFrame):
        y_pred_obj = y_pred_obj.iloc[:, 0]
    y_true = pd.to_numeric(y_true_obj, errors='coerce').to_numpy(dtype=float)
    y_pred = pd.to_numeric(y_pred_obj, errors='coerce').to_numpy(dtype=float)
    return {
        'n': int(np.isfinite(y_true).sum()),
        'MAE': mae(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'MAPE_%': mape(y_true, y_pred),
        'Bias(pred-true)': bias(y_true, y_pred),
    }


def summarize_by_group(df, y_true_col, y_pred_col, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        d = metrics_row(g, y_true_col, y_pred_col)
        rows.append((*keys, d['n'], d['MAE'], d['RMSE'], d['MAPE_%'], d['Bias(pred-true)']))
    cols = list(group_cols) + ['n', 'MAE', 'RMSE', 'MAPE_%', 'Bias(pred-true)']
    return pd.DataFrame(rows, columns=cols)

SITE_MAP = {
    'totalKwh': ('elecTotalKwh', 'site'),
    'totalWater': ('waterM3', 'site'),
    'tempAmb': ('weath_tempAmb', 'site'),
    'value': ('totalOptimValue', 'site'),
}

ZONE_MAP = {
    'totalKwh': ('elecTotalKwh', 'zone'),
    'totalWater': ('waterM3', 'zone'),
    'tempAmb': ('indoorTempDegC', 'zone'),
    'value': ('totalOptimValue', 'zone'),
}

SITE_MAP = {k: v for k, v in SITE_MAP.items() if k in site_base.columns and v[0] in site_base.columns}
ZONE_MAP = {k: v for k, v in ZONE_MAP.items() if k in zone_base.columns and v[0] in zone_base.columns}

site_results = []
site_by_site = {}
for pred_col, (true_col, _) in SITE_MAP.items():
    g = site_base.dropna(subset=['siteId', 'date'])
    g = g[['siteId', 'date', pred_col, true_col]].copy()
    d = metrics_row(g, true_col, pred_col)
    site_results.append({'level': 'site', 'pred_col': pred_col, 'true_col': true_col, **d})
    by = summarize_by_group(g, true_col, pred_col, ['siteId'])
    by.insert(0, 'pred_col', pred_col)
    by.insert(1, 'true_col', true_col)
    site_by_site[pred_col] = by.sort_values('RMSE', ascending=False)

zone_results = []
zone_by_zone = {}
for pred_col, (true_col, _) in ZONE_MAP.items():
    g = zone_base.dropna(subset=['siteId', 'zoneId', 'date'])
    g = g[['siteId', 'zoneId', 'date', pred_col, true_col]].copy()
    d = metrics_row(g, true_col, pred_col)
    zone_results.append({'level': 'zone', 'pred_col': pred_col, 'true_col': true_col, **d})
    by = summarize_by_group(g, true_col, pred_col, ['siteId', 'zoneId'])
    by.insert(0, 'pred_col', pred_col)
    by.insert(1, 'true_col', true_col)
    zone_by_zone[pred_col] = by.sort_values('RMSE', ascending=False)

site_results = pd.DataFrame(site_results).sort_values('RMSE', ascending=False)
zone_results = pd.DataFrame(zone_results).sort_values('RMSE', ascending=False)
summary = pd.concat([site_results, zone_results], ignore_index=True)

display(summary)

,level,pred_col,true_col,n,MAE,RMSE,MAPE_%,Bias(pred-true)
0,site,totalKwh,elecTotalKwh,2598,1.327578e+06,6.537429e+07,5.082400e+03,-1.292057e+06
1,site,totalWater,waterM3,907,7.481480e+04,3.633596e+05,5.248248e+07,7.481479e+04
2,site,tempAmb,weath_tempAmb,3482,8.602488e+00,1.053260e+01,1.484363e+02,-2.826616e+00
3,site,value,totalOptimValue,462,NaN,NaN,NaN,NaN
4,zone,totalWater,waterM3,1698,3.440663e+03,3.738633e+04,3.112393e+07,3.440598e+03
5,zone,totalKwh,elecTotalKwh,16608,1.577626e+02,1.603598e+03,1.547681e+05,1.792943e+01
6,zone,tempAmb,indoorTempDegC,18002,3.075513e+00,6.743173e+00,2.685293e+01,-1.820650e-01
7,zone,value,totalOptimValue,2371,NaN,NaN,NaN,NaN


In [8]:
(summary).to_csv(OUT_DIR / 'metrics_summary_all_metrics_cleaned.csv', index=False)
for metric, df in site_by_site.items():
    df.to_csv(OUT_DIR / f'metrics_by_site_{metric}_cleaned.csv', index=False)
for metric, df in zone_by_zone.items():
    df.to_csv(OUT_DIR / f'metrics_by_zone_{metric}_cleaned.csv', index=False)
for k, log in CLEAN_LOGS.items():
    log.to_csv(OUT_DIR / f'cleanlog_{k}.csv', index=False)
print('wrote to', OUT_DIR)

wrote to output
